In [63]:
import pandas as pd
import numpy as np

In [64]:
def generate_features(n_samples, n_features, prefix):
    df = pd.DataFrame()
    
    classes = ['Antibiotic', 'Antidepressant', 'Painkiller', 'Antihistamine']
    half_life_mapping = {'Short': 1, 'Medium': 2, 'Long': 3}
    
    for i in range(1, n_features + 1):
        if i % 4 == 1:
            df[f'{prefix}_Concentration_{i}'] = np.random.uniform(10.0, 800.0, n_samples)
        elif i % 4 == 2:
            df[f'{prefix}_Prescription_{i}'] = np.random.choice([1, 0], n_samples)
        elif i % 4 == 3:
            df[f'{prefix}_Class_{i}'] = np.random.choice(classes, n_samples)
        else:
            df[f'{prefix}_HalfLife_{i}'] = np.random.choice(list(half_life_mapping.values()), n_samples)
            
    return df


In [65]:
def collision_function(row, p1, p2):
    c1 = row[f'{p1}_Class_3']
    c2 = row[f'{p2}_Class_3']
    conc1 = row[f'{p1}_Concentration_1']
    conc2 = row[f'{p2}_Concentration_1']
    per1 = row[f'{p1}_HalfLife_4']
    per2 = row[f'{p2}_HalfLife_4']
    
    cond1 = (c1 == 'Antibiotic' and c2 == 'Antidepressant' and conc1 > 500)
    cond2 = (per1 == 3 and per2 == 3 and (conc1 + conc2) > 1000)
    cond3 = (c1 == 'Antihistamine' and c2 == 'Antihistamine')
    
    if cond1 or cond2 or cond3:
        return 'Yes'
    return 'No'

In [66]:
def create_dataset(n_samples, n_features_obj1, n_features_obj2):
    obj1_df = generate_features(n_samples, n_features_obj1, 'Drug_A')
    obj2_df = generate_features(n_samples, n_features_obj2, 'Drug_B')
    
    dataset = pd.concat([obj1_df, obj2_df], axis=1)
    dataset['Collision'] = dataset.apply(lambda r: collision_function(r, 'Drug_A', 'Drug_B'), axis=1)
    
    return dataset


In [67]:
sample_sizes = [50, 250, 750, 1500]
feature_counts = [5, 9, 15]
datasets = {}
dataset_id = 1

for n_samples in sample_sizes:
    for n_features in feature_counts:
        df = create_dataset(n_samples, n_features, n_features)
        name = f"ID:{dataset_id}_V{n_samples}_F{n_features}"
        datasets[name] = df
        dataset_id += 1
        df.to_csv(name)

for ds in datasets:
    print(ds)

print(datasets["ID:3_V50_F15"].head())

ID:1_V50_F5
ID:2_V50_F9
ID:3_V50_F15
ID:4_V250_F5
ID:5_V250_F9
ID:6_V250_F15
ID:7_V750_F5
ID:8_V750_F9
ID:9_V750_F15
ID:10_V1500_F5
ID:11_V1500_F9
ID:12_V1500_F15
   Drug_A_Concentration_1  Drug_A_Prescription_2  Drug_A_Class_3  \
0              408.204813                      0   Antihistamine   
1              612.724605                      1      Antibiotic   
2              198.553533                      0  Antidepressant   
3              707.204211                      0   Antihistamine   
4              619.540138                      1      Antibiotic   

   Drug_A_HalfLife_4  Drug_A_Concentration_5  Drug_A_Prescription_6  \
0                  1               34.559415                      1   
1                  3              405.558317                      0   
2                  1              366.952095                      1   
3                  2              700.071302                      0   
4                  2              446.711550                      0   

 